In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import equinox as eqx
from flowjax.flows import coupling_flow
from flowjax.distributions import Normal
from flowjax.bijections import Affine, RationalQuadraticSpline

from types import SimpleNamespace

import matplotlib.pyplot as plt


In [ ]:
DEFAULT_CONFIG = SimpleNamespace(
    param_dim=2, # how many parameters do we want to infer?
    freqs=jnp.linspace(1e-5, 1e-3, 256)
)
DEFAULT_CONFIG.cond_dim = len(DEFAULT_CONFIG.freqs)

# Build the signal model

I have defined below a simplified signal model, but the noise is missing. 
Remember in jax, you can draw from a normal distribution via the example below. 

In [ ]:
samples = jax.random.normal(jax.random.key(0), shape=(10_000))

In [ ]:
plt.hist(samples, bins=30, density=True, histtype="step");

In [ ]:
def get_noise(freqs, key, noise_level=1.0):
    return None

def get_cutoff_freq(mass):
    return 4400 / mass

def get_asd(freqs):
    return freqs ** (- 1)

def get_signal(freqs, precession_amplitude, log10_mass):

    # GW signal
    amplitude = 1e-1
    characteristic_freq = 1e-5
    
    mass = 10 ** log10_mass
    cutoff_freq = get_cutoff_freq(mass=mass)

    signal = freqs ** (-7 / 6) * (
        1 + precession_amplitude * jnp.sin(freqs / characteristic_freq)
    ) * 1 / (1 + jnp.exp((freqs - cutoff_freq)))

    return amplitude * signal

def get_signal_whitened(freqs, precession_amplitude, log10_mass):
    
    asd = get_asd(freqs)
    signal = get_signal(freqs, precession_amplitude, log10_mass)
    
    return signal / asd

In [ ]:
signal_whitened = ...
noise = ...
data_whitened = signal_whitened + noise

# let's plot the whitened data and true signal
plt.plot(DEFAULT_CONFIG.freqs, ...)
plt.plot(DEFAULT_CONFIG.freqs, ...)
plt.xlabel("Frequency (Hz)")
plt.xscale("log")
plt.yscale("log")
plt.ylim(7e-4, None)

# Prepare data generation

In [ ]:
# reminder: draw from a uniform distribution
samples = jax.random.uniform(jax.random.key(1), shape=(10_000))

In [ ]:
plt.hist(samples, bins=30, density=True, histtype="step");

In [ ]:
def draw_params_from_prior(key):

    key1, key2 = jax.random.split(key)

    # from from prior
    precession_amplitude = ...
    log10_mass = ...

    params = dict(
        precession_amplitude=precession_amplitude,
        log10_mass=log10_mass
    )
    return params

def data_generator(key, config=None):

    if config is None:
        config = DEFAULT_CONFIG

    params = ... # draw from prior
    
    freqs = config.freqs

    # compute whitened signal
    signal_whitened = ...

    noise_key = jax.random.split(key)[0]
    noise_whitened = ...

    data = signal_whitened + noise_whitened

    params = jnp.concatenate([params['precession_amplitude'], params['log10_mass']])

    return params, data


# Setting up the normalizing flow and associated neural networks

In [ ]:
DEFAULT_CONFIG.__dict__.update({
    "flow_layers": 4,
    "nn_width": 64,
    "nn_depth": 4,
})

In [ ]:
def construct_flow(config, flow_key):
    return coupling_flow(
        flow_key,
        base_dist=Normal(jnp.zeros(config.param_dim)),
        cond_dim=config.cond_dim,
        transformer=Affine(),
        flow_layers=config.flow_layers,
        nn_width=config.nn_width,
        nn_depth=config.nn_depth,
    )

flow_key = jax.random.key(0)
model = construct_flow(DEFAULT_CONFIG, flow_key)

In [ ]:
# let's try to evaluate the log probability
# two ingredients are needed: parameters and conditions
model.log_prob(
    x=...,
    condition=...
)

Let's plot some samples. Why do the samples look weird?

In [ ]:
import numpy as np
import corner

samples = model.sample(
    key=...,
    sample_shape=...,
    condition=jnp.zeros(DEFAULT_CONFIG.cond_dim)
)

corner.corner(np.array(samples));

# Defining the standard SBI loss and training loop

The standard SBI loss is 
$$
{\rm loss} = -\frac{1}{N}\sum_{i=1}^N q(x_i | d_i)
$$
that we want to minimize. 

In [ ]:

def get_loss(model, params, data):
    log_prob = model.log_prob(
        x=...,
        condition=...,
    )
    return ...

def train_epoch(model, opt_state, data_generator, key):

    params, data = data_generator(key)

    loss, gradient = eqx.filter_value_and_grad(get_loss)(model, params, data)
    updates, new_opt_state = optimizer.update(gradient, opt_state, eqx.filter(model, eqx.is_inexact_array))

    new_model = eqx.apply_updates(model, updates)

    return new_model, new_opt_state, loss

def test_epoch(model, data_generator, key):

    test_key = jax.random.split(key)[0]
    params, data = data_generator(test_key)
    loss = get_loss(model, params, data)
    return loss
    

# Training

In [ ]:
import optax
import tqdm

optimizer = optax.adam(learning_rate=1e-4)
opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

train_losses = []
test_losses = []

flow_keymodel = construct_flow(DEFAULT_CONFIG, jax.random.key(120))

for i in tqdm.tqdm(range(100)):
    model, opt_state, loss = train_epoch(model, opt_state, data_generator, jax.random.key(i))
    train_losses.append(loss)

    test_loss = test_epoch(model, data_generator, jax.random.key(i))
    test_losses.append(test_loss)

plt.plot(train_losses)
plt.plot(test_losses)

This training is very slow, try compiling it by add `@eqx.filter_jit` to the train functions above. 
It should be faster now. Now we can train for longer! 

The loss is very noise. How can we solve this?

In [ ]:
params, data = data_generator(jax.random.key(1))
samples = model.sample(
    key=...,
    sample_shape=...,
    condition=...
)

# reminder: corner likes np.arrays
corner.corner(
    np.array(samples),
    truths=...,
);

Let's improve the training process. Can we improve the gradient?

In [ ]:
# vmap is a useful function
key = jax.random.key(0)
batch_size = ...
keys = jax.random.split(key, num=batch_size)

params_batch, data_batch = jax.vmap(data_generator)(keys)


In [ ]:
@eqx.filter_jit
def train_epoch_batch(model, opt_state, data_generator, key, batch_size=16):
    
    params_batch, data_batch = ...

    loss, gradient = eqx.filter_value_and_grad(get_loss)...
    updates, new_opt_state = optimizer.update(gradient, opt_state, eqx.filter(model, eqx.is_inexact_array))

    new_model = eqx.apply_updates(model, updates)

    return new_model, new_opt_state, loss

In [ ]:
model = construct_flow(DEFAULT_CONFIG, flow_key)
optimizer = optax.adam(learning_rate=2e-4)
opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

train_losses = []
test_losses = []
for i in tqdm.tqdm(range(1000)):
    model, opt_state, loss = train_epoch_batch(...)
    train_losses.append(loss)

    test_loss = test_epoch(model, data_generator, jax.random.key(i))
    test_losses.append(test_loss)

In [ ]:
# let's plot the train and tess losses

# Inference

Draw random signal and produce posterior samples

In [ ]:
params, data = data_generator(key=...)
true_signal = get_signal_whitened(DEFAULT_CONFIG.freqs, precession_amplitude=..., log10_mass=...)

samples = model.sample(
    key=...,
    sample_shape=...,
    condition=...
)

corner.corner(
    np.array(samples),
    truths=np.array(params),
);

In [ ]:
signals = ... # let's try to get our signals via vmap from the posterior samples

perc = jnp.percentile(signals, jnp.array([5, 50, 95]), axis=0)

plt.plot(DEFAULT_CONFIG.freqs, perc[1], label="Median")
plt.fill_between(DEFAULT_CONFIG.freqs, perc[0], perc[2], alpha=0.5, label="90% CI")
plt.plot(DEFAULT_CONFIG.freqs, abs(data))
plt.plot(DEFAULT_CONFIG.freqs, true_signal)
plt.yscale("log")
plt.ylim(1e-2, None)
plt.xscale("log")

In [ ]:
# save model parameters
# eqx.tree_serialise_leaves("../models/flow_2D_simple.eqx", model)